# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip -q install duckdb huggingface_hub


In [5]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [13]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents the daily performance of a single piece of content (unique content_id + date).
Time window: We are evaluating a mid-panel month, strictly spanning from 2026-03-01 to 2026-03-31.

In [18]:
# Fetch just 1 row to see all the column names
sample_df = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(sample_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [19]:
span_check = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

print("--- Date Span & Row Count ---")
display(span_check)

grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) as row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()


print(f"Grain violations (should be 0): {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Date Span & Row Count ---


,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Grain Check ---
Grain violations (should be 0): 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_avg_position, sessions_organic, ga4_pageviews (We will use historical, backward-looking aggregations of these metrics).
Label: Future gsc_clicks (e.g., total clicks accumulated in the 7 or 30 days following the report_date).
Context: content_hash_id, client_hash_id, report_date (These define the grain and allow us to join tables or group data, but the model won't train directly on the raw IDs).
Excluded: month (Redundant partition column since we parse time from report_date), and the entire fact_daily_sample table (Excluded because the instructions strictly forbid using the final month for label development; it must remain sealed).

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Verify availability (How many rows survive the IS TRUE filter?)
availability_check = con.sql(f"""
    SELECT COUNT(*) as surviving_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
""").df()

print("--- Availability Check (GSC Data Present) ---")
display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Availability Check (GSC Data Present) ---


,surviving_rows
0,3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Claims:

Missing Values: We verify the null rates for our primary features (gsc_impressions, ga4_pageviews) to ensure data quality.

Windows: We prove the time-travel logic by using window functions to calculate a backward-looking feature (past_7d_impressions) and a forward-looking label (future_7d_clicks) without leaking future data into the present.

In [21]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS null_pageviews,
        SUM(CASE WHEN sessions_organic IS NULL THEN 1 ELSE 0 END) AS null_organic_sessions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
""").df()

display(missing_check)



window_demo = con.sql(f"""
    WITH daily_data AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_clicks,
            gsc_impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-20'
          AND gsc_data_available IS TRUE
    )
    SELECT
        content_hash_id,
        report_date,
        gsc_clicks AS current_day_clicks,

        -- FEATURE: Look backward (Strictly prior 7 days)
        SUM(gsc_impressions) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS past_7d_impressions,

        -- LABEL / TRAP: Look forward (Next 7 days, excluding today)
        SUM(gsc_clicks) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING
        ) AS future_7d_clicks

    FROM daily_data
    ORDER BY content_hash_id, report_date
    LIMIT 15
""").df()

display(window_demo)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,null_impressions,null_pageviews,null_organic_sessions
0,3611061,0.0,1528366.0,1528366.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,current_day_clicks,past_7d_impressions,future_7d_clicks
0,content_000005d4ced12088,2026-03-03,0,NaN,0.0
1,content_000005d4ced12088,2026-03-04,0,1.0,0.0
2,content_000005d4ced12088,2026-03-05,0,5.0,0.0
3,content_000005d4ced12088,2026-03-06,0,7.0,0.0
4,content_000005d4ced12088,2026-03-10,0,10.0,0.0
5,content_000005d4ced12088,2026-03-11,0,11.0,0.0
6,content_000005d4ced12088,2026-03-13,0,14.0,0.0
7,content_000005d4ced12088,2026-03-14,0,17.0,0.0
8,content_000005d4ced12088,2026-03-15,0,20.0,0.0
9,content_000005d4ced12088,2026-03-16,0,18.0,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitation: This slice can never tell us the complete on-page engagement (GA4 metrics) for every successful search impression (GSC metrics). Because clients adopt tracking tools at different times, there is a significant volume of rows where Google Search Console data is present, but Google Analytics 4 data is entirely missing. We cannot safely assume a zero in GA4 means "no engagement"—it often just means "no tracking."

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

limitation_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS tracking_blind_spot
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

gsc_count = limitation_check['rows_with_gsc'][0]
blind_spot_count = limitation_check['tracking_blind_spot'][0]
drop_off_pct = (blind_spot_count / gsc_count) * 100 if gsc_count > 0 else 0

print(f"--- Data Limitation: Tracking Asymmetry ---")
display(limitation_check)
print(f"\nConclusion: {drop_off_pct:.1f}% of the rows with Search data (GSC) are completely blind to on-page engagement (GA4).")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Data Limitation: Tracking Asymmetry ---


,total_march_rows,rows_with_gsc,rows_with_ga4,tracking_blind_spot
0,9841378,3611061.0,413966.0,1718348.0



Conclusion: 47.6% of the rows with Search data (GSC) are completely blind to on-page engagement (GA4).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.